# GEC Toolkit Notebook

Implements the Generalized Efficiency Coefficient (GEC) and Compression Scaling Kernel (CSK):
- `GEC0 = (Y/X) / Cmax`
- CSK geometric aggregate `λ(κ)` for diagnostics
- Delta-method variance (independent & joint)
- Frontier audit & re-normalization

Run cells top-to-bottom.

In [ ]:
import math
from typing import Dict, Iterable, Optional, List

def _safe_div(a: float, b: float) -> float:
    return float('nan') if b == 0 else a / b

def gec0(Y: float, X: float, Cmax: float) -> float:
    """Return normalized efficiency GEC0 = (Y/X) / Cmax. Returns NaN if invalid."""
    eta = _safe_div(Y, X)
    if math.isnan(eta) or Cmax <= 0:
        return float('nan')
    return eta / Cmax

def _geom_mean(values: Iterable[float], weights: Optional[Iterable[float]] = None) -> float:
    vals = list(values)
    if any(v <= 0 for v in vals):
        return float('nan')
    if weights is None:
        w = [1.0/len(vals)] * len(vals)
    else:
        w = list(weights)
        s = sum(w)
        if s <= 0:
            return float('nan')
        w = [wi/s for wi in w]
    log_sum = 0.0
    for v, wi in zip(vals, w):
        log_sum += wi * math.log(v)
    return math.exp(log_sum)

def csk_lambda(kappa: Dict[str, float], weights: Optional[Dict[str, float]] = None) -> float:
    """Geometric aggregate λ(κ); expected keys: S,H,D,R,E in [0,1]."""
    keys = list(kappa.keys())
    vals = [max(1e-9, min(1.0, float(kappa[k]))) for k in keys]
    if weights is None:
        w = None
    else:
        w = [float(weights.get(k, 0.0)) for k in keys]
    return _geom_mean(vals, w)

def gec0_variance_delta(Y: float, X: float, Cmax: float, varY: float, varX: float, varC: float) -> float:
    """Independent delta-method variance (Eq. 1)."""
    if X <= 0 or Cmax <= 0:
        return float('nan')
    term1 = (1.0 / (Cmax * X))**2 * varY
    term2 = (Y / (Cmax * X**2))**2 * varX
    term3 = (Y / (X * Cmax**2))**2 * varC
    return term1 + term2 + term3

def gec0_variance_delta_joint(
    Y: float, X: float, Cmax: float,
    varY: float, varX: float, varC: float,
    cov_YX: float = 0.0, cov_YC: float = 0.0, cov_XC: float = 0.0
) -> float:
    """Full delta method with covariance terms (Eq. 2)."""
    if X <= 0 or Cmax <= 0:
        return float('nan')
    dY = 1.0 / (X * Cmax)
    dX = -Y / (X**2 * Cmax)
    dC = -Y / (X * Cmax**2)
    var = (dY**2) * varY + (dX**2) * varX + (dC**2) * varC
    var += 2.0 * dY * dX * cov_YX
    var += 2.0 * dY * dC * cov_YC
    var += 2.0 * dX * dC * cov_XC
    return var

def frontier_audit_and_renorm(series: List[dict], Cnew: float) -> List[dict]:
    """Recompute GEC0 with new frontier Cnew; flag >1.0; log adj factor Cold/Cnew."""
    if Cnew <= 0:
        raise ValueError("Cnew must be > 0")
    out = []
    for rec in series:
        Y = float(rec.get("Y", float("nan")))
        X = float(rec.get("X", float("nan")))
        Cold = float(rec.get("Cold", float("nan")))
        g_new = gec0(Y, X, Cnew)
        adj = float("nan") if Cold <= 0 else (Cold / Cnew)
        flag = bool(g_new > 1.0) if not math.isnan(g_new) else False
        rec2 = dict(rec)
        rec2["GEC0_new"] = g_new
        rec2["adj_factor"] = adj
        rec2["flag_error"] = flag
        out.append(rec2)
    return out


In [ ]:
# Quick start examples
comm = dict(Y=3.0, X=1.0, Cmax=3.46)           # Communications @ 10 dB (illustrative)
thermo = dict(Y=0.3244, X=1.0, Cmax=0.79)       # Thermo (Carnot bound, illustrative)
ml = dict(Y=0.843, X=1.0, Cmax=0.905)           # ML (ImageNet normalized to SOTA, illustrative)

for name, d in {"comm":comm, "thermo":thermo, "ml":ml}.items():
    G0 = gec0(d["Y"], d["X"], d["Cmax"])
    print(name, "GEC0=", round(G0, 4))

# CSK example
kappa = {"S":0.95,"H":0.80,"D":0.90,"R":0.85,"E":0.90}
lam = csk_lambda(kappa)
print("λ(κ) =", round(lam, 4), "→ composite (diagnostic) =", round(lam * gec0(**comm), 4))

In [ ]:
# Frontier audit demo
series = [
    {"id":"A","Y":3.00,"X":1.00,"Cold":3.30},
    {"id":"B","Y":3.10,"X":1.00,"Cold":3.30},
    {"id":"C","Y":2.95,"X":1.00,"Cold":3.30},
]
Cnew = 3.46
audited = frontier_audit_and_renorm(series, Cnew)
audited

In [ ]:
# Plots – each on its own figure; using matplotlib (no seaborn)
import matplotlib.pyplot as plt

examples = {
    "communications": dict(Y=3.0, X=1.0, Cmax=3.46, kappa={"S":.95,"H":.80,"D":.90,"R":.85,"E":.90}),
    "thermo_carnot": dict(Y=0.3244, X=1.0, Cmax=0.79, kappa={"S":.98,"H":.70,"D":.90,"R":.95,"E":.59}),
    "thermo_exergy": dict(Y=0.3244, X=1.0, Cmax=0.59, kappa={"S":.98,"H":.70,"D":.90,"R":.95,"E":.59}),
    "ml_imagenet":  dict(Y=0.843, X=1.0, Cmax=0.905, kappa={"S":.90,"H":.80,"D":.70,"R":.85,"E":.95}),
    "finance_roa":  dict(Y=0.012, X=1.0, Cmax=0.020, kappa={"S":.85,"H":.80,"D":.60,"R":.70,"E":.75}),
    "governance":   dict(Y=0.80, X=1.0, Cmax=0.95,  kappa={"S":.95,"H":.80,"D":.75,"R":.90,"E":.90}),
}

labels = list(examples.keys())
G0 = [gec0(**examples[k]) for k in labels]
Gc = []
for k in labels:
    lam = csk_lambda(examples[k]["kappa"])
    Gc.append(G0[labels.index(k)] * lam)

# Plot GEC0
plt.figure()
plt.bar(range(len(labels)), G0)
plt.xticks(range(len(labels)), labels, rotation=30, ha='right')
plt.ylabel("GEC0")
plt.title("Normalized Efficiency by Example")
plt.tight_layout()
plt.show()

# Plot composite (diagnostic only)
plt.figure()
plt.bar(range(len(labels)), Gc)
plt.xticks(range(len(labels)), labels, rotation=30, ha='right')
plt.ylabel("GEC0 · λ(κ)  [diagnostic]")
plt.title("Composite Score (Diagnostics)")
plt.tight_layout()
plt.show()

In [ ]:
# Fixed-point map: t_{n+1} = k + 1/t_n → metallic mean fixed point (φ for k=1)
import math
import matplotlib.pyplot as plt

def metallic_fixed_point(k: float) -> float:
    return 0.5 * (k + math.sqrt(k*k + 4.0))

def iterate_map(k: float, t0: float, steps: int = 25):
    t = t0
    seq = [t]
    for _ in range(steps):
        t = k + 1.0/max(1e-12, t)
        seq.append(t)
    return seq

# Golden ratio (k=1)
plt.figure()
for t0 in [0.6, 1.0, 2.4, 3.5]:
    seq = iterate_map(1.0, t0, 25)
    plt.plot(seq, label=f"t0={t0}")
plt.axhline(metallic_fixed_point(1.0), linestyle="--")
plt.xlabel("Iteration"); plt.ylabel("t_n"); plt.title("Convergence to φ (k=1)")
plt.tight_layout(); plt.show()

# Silver ratio (k=2)
plt.figure()
for t0 in [0.6, 1.0, 2.4, 3.5]:
    seq = iterate_map(2.0, t0, 25)
    plt.plot(seq, label=f"t0={t0}")
plt.axhline(metallic_fixed_point(2.0), linestyle="--")
plt.xlabel("Iteration"); plt.ylabel("t_n"); plt.title("Convergence to Silver Ratio (k=2)")
plt.tight_layout(); plt.show()